## 05 - Aperiodic exponent and spectral entropy

For recordings grouped by condition: the aperiodic (1/f) exponent and spectral entropy from the per-epoch table, the aperiodic figure (`figAperiodic`, panels selectable), the summary inset (`figSummaryInset`) and statistics with a false-discovery correction across the two measures (`fdrAcrossMeasures`).

Writes `cache/<animal>/csvs/`, `cache/epochs/`, and under `outputs/05_aperiodic_exponent_and_entropy/` both figures, `anova.csv`, `pairwise.csv`, `per_animal_primary_window.csv`, `fit_quality.csv` and `run_info.json`.

### Variables

In [ ]:
EDF_FOLDER = "../../Completed_Analyses/Emergence/raw_edfs"  # folder holding the .edf files (relative to this notebook)
RECORDINGS = [  # one dict per recording: animal_id, condition (as recorded), recording_number, edf, channel_index, events (recording-clock s)
    dict(animal_id="PEEG01M", condition="AwakeControl", recording_number=1, edf="EmergRev_3Perc_AwakeControl_PEEG01M_2026-09-10_13_30_25_export.edf", channel_index=0, events={"onset": 0}),  # 0
    dict(animal_id="PEEG01M", condition="Induction", recording_number=2, edf="EmergRev_3Perc_Induction_PEEG01M_2026-09-01_13_23_07_export.edf", channel_index=0, events={"onset": 1800}),  # 1: 2026-09-01 session
    dict(animal_id="PEEG02M", condition="AwakeControl", recording_number=1, edf="EmergRev_3Perc_AwakeControl_PEEG02M_2026-09-10_13_30_25_export.edf", channel_index=0, events={"onset": 0}),  # 2
    dict(animal_id="PEEG02M", condition="Induction", recording_number=1, edf="EmergRev_3Perc_Induction_PEEG02M_2026-08-27_15_49_57_export.edf", channel_index=0, events={"onset": 1800}),  # 3
    dict(animal_id="PEEG03F", condition="AwakeControl", recording_number=1, edf="EmergRev_3Perc_AwakeControl_PEEG03F_2026-09-10_13_30_25_export.edf", channel_index=1, events={"onset": 0}),  # 4: channel 1
    dict(animal_id="PEEG03F", condition="Induction", recording_number=1, edf="EmergRev_3Perc_Induction_PEEG03F_2026-08-27_15_49_57_export.edf", channel_index=1, events={"onset": 1800}),  # 5: channel 1
    dict(animal_id="PEEG04F", condition="AwakeControl", recording_number=1, edf="EmergRev_3Perc_AwakeControl_PEEG04F_2026-09-10_13_30_25_export.edf", channel_index=0, events={"onset": 0}),  # 6
    dict(animal_id="PEEG04F", condition="Induction", recording_number=1, edf="EmergRev_3Perc_Induction_PEEG04F_2026-09-15_13_46_21_export.edf", channel_index=0, events={"onset": 1860}),  # 7: onset 1860
    dict(animal_id="PEEG01M", condition="Alone", recording_number=1, edf="EmergRev_3Perc_Alone_PEEG01M_2026-09-10_14_37_59_export.edf", channel_index=0, events={"onset": 900}),  # 8: 0.3% alone, onset 900
    dict(animal_id="PEEG02M", condition="Alone", recording_number=1, edf="EmergRev_3Perc_Alone_PEEG02M_2026-08-31_14_51_28_export.edf", channel_index=0, events={"onset": 300}),  # 9: 0.3% alone, onset 300
    dict(animal_id="PEEG03F", condition="Alone", recording_number=1, edf="EmergRev_3Perc_Alone_PEEG03F_2026-08-31_14_51_28_export.edf", channel_index=1, events={"onset": 300}),  # 10: 0.3% alone, channel 1, onset 300 (as in all reported results)
    dict(animal_id="PEEG04F", condition="Alone", recording_number=1, edf="EmergRev_3Perc_Alone_PEEG04F_2026-09-10_14_37_59_export.edf", channel_index=0, events={"onset": 900}),  # 11: 0.3% alone, onset 900
    dict(animal_id="PEEG01M", condition="Induction", recording_number=1, edf="EmergRev_3Perc_Induction_PEEG01M_2026-08-26_15_28_31_export.edf", channel_index=0, events={"onset": 1800}),  # 12: 2026-08-26 session: a second PEEG01M Induction session
]
CONDITIONS = {  # {condition label: [(RECORDINGS index or an inline dict, channel analysed), ...]}, one entry per animal; key order = plotting order
    "AwakeControl": [(0, 0), (2, 0), (4, 1), (6, 0)],  # awake, 0% isoflurane
    # "No Induction": [(8, 0), (9, 0), (10, 1), (11, 0)],  # uncomment for the 0.3%-alone condition (a third group)
    "Induction": [(1, 0), (3, 0), (5, 1), (7, 0)],  # 0.3% after induction
}
ONSET_EVENT = "onset"  # event at 0 on the epoch time axis; every recording needs it in its events
EPOCH_S = 30  # epoch length (s)
WELCH_WINDOW_S = 4  # Welch sub-window inside an epoch (s); frequency resolution = 1 / WELCH_WINDOW_S
FIT_RANGE = (1.0, 20.0)  # Hz range of the aperiodic fit, spectral entropy and median frequency
EPOCH_BANDS = {"delta": (1.5, 4.0), "theta": (4.0, 8.0), "alpha": (8.0, 13.0)}  # Hz bands summarised per epoch (<band>_abs, _rel, _periodic)
WINDOWS = {"30-55 min": (1800, 3350), "40-55 min": (2400, 3300), "whole hour 0-55 min": (0, 3350)}  # analysis windows, seconds after ONSET_EVENT (recordings must cover them)
PRIMARY = "30-55 min"  # window shown in the figures (a key of WINDOWS)
MEASURES = ["exponent", "spectral_entropy"]  # the two reported measures: tested together and corrected across measures
PANELS = "abc"  # panels of figAperiodic: a = mean spectra with fitted backgrounds, b = exponent over time, c = per-animal exponent
INSET_PANELS = (("exponent", "Aperiodic exponent", "Background slope"),
                ("spectral_entropy", "Spectral entropy (0-1)", "Spectral entropy"))  # (column, y label, title) per panel of the summary inset
EXPONENT_YLIM = (-0.15, 2)  # y range of the exponent panel of the inset
FDR_METHOD = "fdr_bh"  # correction across MEASURES for each condition pair (two conditions: one contrast per measure)
P_COL = "q_measures"  # p-value column drawn in the inset: "q_measures" (across measures) or "p_adj" (across condition pairs, for three conditions)
FIT_QUALITY = ["fit_r2", "fit_error"]  # aperiodic fit quality columns added to the per-animal table (not tested)
P_ADJUST = "fdr_bh"  # correction of the paired t-tests across condition pairs within a metric and window
OVERWRITE_RAW = False  # True re-extracts the raw CSVs
OVERWRITE_EPOCHS = False  # True refits the epoch table even when the cache matches design and settings
WORK_DIR = "cache"  # processing cache (raw CSVs, epoch table), reused between runs and by notebook 04
OUTPUT_DIR = "outputs/05_aperiodic_exponent_and_entropy"  # figures and tables; files are never overwritten
FIGURE_FORMATS = ("png", "svg")  # one file per format for each figure
DPI = 300  # resolution of the png
BBOX_INCHES = None  # None saves the exact canvas (manuscript style); "tight" crops to the content so no title is clipped

In [ ]:
from pathlib import Path  # path handling
import sys  # module search path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ecog").is_dir())  # repo root = first parent folder containing ecog/
sys.path.insert(0, str(ROOT))  # makes `import ecog` work without installing anything
import os  # path normalising
EDF_DIR = Path(os.path.abspath(EDF_FOLDER))  # absolute EDF folder with any `..` collapsed (relative paths start at this notebook's folder)
import mne  # EDF reader used by ecog
mne.set_log_level("WARNING")  # hides the EDF reader's progress messages
import contextlib, io, importlib  # output control and dynamic import
with contextlib.redirect_stderr(io.StringIO()):  # fooof prints a deprecation notice to stderr when it is first imported
    importlib.import_module("fooof")  # imported once here, before ecog needs it
from ecog import (Recording, extractRecording, buildEpochTable, compareAcrossConditions, fdrAcrossMeasures, perAnimalMeans, fitQuality,
                  saveFigure, saveTable, writeRunInfo)  # processing, epoch table, statistics, saving
from ecog.plots import figAperiodic, figSummaryInset  # aperiodic decomposition figure and compact summary inset
WORK = Path(WORK_DIR)  # cache folder
OUT = Path(OUTPUT_DIR)  # output folder

### 1. Recordings grouped by condition

In [ ]:
def toRecording(entry):  # (RECORDINGS index or dict, channel) -> (Recording, channel)
    spec, channel = entry  # recording description and the channel analysed
    spec = RECORDINGS[spec] if isinstance(spec, int) else spec  # an int picks from RECORDINGS, a dict is used as given
    return Recording(edf_folder=EDF_DIR, **{**spec, "channel_index": channel}), channel  # the channel in CONDITIONS decides what is extracted


# {condition label: [(Recording, channel), ...]}, the input of every function below
batches = {label: [toRecording(e) for e in entries] for label, entries in CONDITIONS.items()}
ORDER = list(CONDITIONS)  # condition order in figures and tests
for entries in batches.values():  # check every file, channel and event before processing
    for rec, _ in entries:
        rec.validate()  # stops with one message listing every problem
{n: len(v) for n, v in batches.items()}  # recordings per condition

### 2. EDF to raw CSV (skipped when cached)

In [ ]:
for entries in batches.values():  # every recording of every condition
    for rec, _ in entries:
        extractRecording(rec, WORK, overwrite=OVERWRITE_RAW)  # raw CSV of the channel; skipped when it exists

### 3. Per-epoch table (cached in cache/epochs, shared with notebook 04)

In [ ]:
# per-epoch table (aperiodic fit, band measures, state metrics) and spectra; reloaded from cache/epochs when design and settings are unchanged
table, spectra = buildEpochTable(batches, WORK, cache=WORK / "epochs", force=OVERWRITE_EPOCHS, onset_event=ONSET_EVENT,
                                 epoch_s=EPOCH_S, welch_window_s=WELCH_WINDOW_S, fit_range=FIT_RANGE, bands=EPOCH_BANDS)
print(len(table), "epochs;", table["recording"].nunique(), "recordings")  # size of the table
fitQuality(table, WINDOWS[PRIMARY], ORDER)  # aperiodic fit quality in the primary window: R2 and error, mean/SD across animals, median/min over epochs

### 4. Statistics for the two measures, with a correction across measures

In [ ]:
# repeated-measures ANOVA and paired t-tests per animal for MEASURES in every window
anova, pairwise = compareAcrossConditions(table, MEASURES, WINDOWS, ORDER, p_adjust=P_ADJUST)
pairwise = fdrAcrossMeasures(pairwise, PRIMARY, metrics=MEASURES, method=FDR_METHOD)  # adds q_measures: correction across MEASURES in the primary window
pairwise[pairwise["window"] == PRIMARY][["metric", "condition_1", "condition_2", "n", "mean_1", "mean_2", "t", "p", "p_adj", "q_measures"]].round(4)  # primary-window contrasts

### 5. Aperiodic figure (choose panels with PANELS)

In [ ]:
# mean spectra + fitted backgrounds (a), exponent time course (b), paired per-animal exponent (c)
fig = figAperiodic(table, spectra, WINDOWS[PRIMARY], ORDER, anova=anova, window_label=PRIMARY, fit_range=FIT_RANGE, panels=PANELS)
saveFigure(fig, OUT / "aperiodic_exponent", dpi=DPI, bbox_inches=BBOX_INCHES, formats=FIGURE_FORMATS)  # canvas kept unless BBOX_INCHES = "tight"

### 6. Summary inset: exponent and spectral entropy with significance brackets

In [ ]:
# compact paired panels; brackets read the p-values from column P_COL of `pairwise`
fig = figSummaryInset(table, WINDOWS[PRIMARY], ORDER, anova, pairwise, PRIMARY, p_col=P_COL, panels=INSET_PANELS, exponent_ylim=EXPONENT_YLIM)
saveFigure(fig, OUT / "summary_inset", dpi=DPI, bbox_inches=BBOX_INCHES, formats=FIGURE_FORMATS)  # canvas kept unless BBOX_INCHES = "tight"

### 7. Save tables

In [ ]:
perAnimal = perAnimalMeans(table, MEASURES + FIT_QUALITY, WINDOWS[PRIMARY], ORDER)  # each animal's mean per condition in the primary window, with fit quality
paths = [saveTable(anova, OUT / "anova.csv"), saveTable(pairwise, OUT / "pairwise.csv"),
         saveTable(perAnimal, OUT / "per_animal_primary_window.csv"),
         saveTable(fitQuality(table, WINDOWS[PRIMARY], ORDER), OUT / "fit_quality.csv")]  # statistics, per-animal values and fit quality, never overwritten
info = writeRunInfo(OUT, settings=dict(conditions={k: len(v) for k, v in CONDITIONS.items()}, windows=WINDOWS, primary=PRIMARY, measures=MEASURES,
                                       p_adjust=P_ADJUST, fdr_method=FDR_METHOD, p_col=P_COL, epoch_s=EPOCH_S, fit_range=FIT_RANGE,
                                       onset_event=ONSET_EVENT))  # versions and settings
print(paths, info)  # files written